In [ ]:
# =========================================================
# Full Professional Medical AI Chat Script (LangChain + Transformers)
# =========================================================

!pip install -q --upgrade transformers optimum auto-gptq
!pip install -q langchain langchain-community
!pip install fastapi uvicorn pyngrok nest_asyncio

# ------------------ Imports ------------------
import os
import re
import threading
from collections import defaultdict
from typing import Dict, List

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok

# ------------------ Port Cleanup ------------------
PORT = 8000
try:
    os.system(f"fuser -k {PORT}/tcp")
except Exception:
    pass

# ------------------ Model Loading ------------------
MODEL_ID = "fady-50/F-Chat-Model-GPTQ"
USE_FP16 = True

print("⏳ Loading tokenizer & model...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=(torch.float16 if USE_FP16 else torch.float32),
    low_cpu_mem_usage=True,
    trust_remote_code=True
)

gen_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.1,
    do_sample=False,
    repetition_penalty=1.2,
    return_full_text=False
)

print("✅ Model ready")

# ------------------ Session Storage ------------------
MAX_QUESTIONS = 8
CONFIDENCE_THRESHOLD = 0.85  # Stop early if confident

store: Dict[str, List[Dict[str, str]]] = defaultdict(list)
question_counters: Dict[str, int] = defaultdict(int)

def get_question_count(session_id: str) -> int:
    return question_counters[session_id]

def increment_question_count(session_id: str):
    question_counters[session_id] += 1

# ------------------ Cleaning Helper ------------------
def strict_clean_response(text: str) -> str:
    if not text:
        return "Could you describe your symptoms in more detail?"
    raw = text
    if "[/INST]" in raw:
        raw = raw.split("[/INST]")[-1]
    raw = raw.replace("Doctor:", "").replace("AI:", "").strip()
    if "READY_TO_DIAGNOSE" in raw:
        return "READY_TO_DIAGNOSE"
    m = re.search(r".*?\?", raw)
    if m:
        return m.group().strip()
    return raw.split("\n")[0].strip()

# ------------------ Report Extraction ------------------
def extract_report_data(text: str) -> Dict[str, str]:
    txt = text.strip()
    data = {
        "diagnosis": "Pending Diagnosis",
        "immediate_action": "Visit a doctor immediately",
        "self_care": "Rest and monitor symptoms carefully",
        "severity": "Unknown",
        "specialist": "General Practitioner",
        "advice": "Stay hydrated and avoid triggers",
        "when_to_seek_help": "Seek urgent care if symptoms worsen",
        "cause": "Under investigation"
    }

    patterns = {
        "diagnosis": r"Disease:\s*(.*)",
        "immediate_action": r"Immediate Action:\s*(.*)",
        "self_care": r"Self-Care:\s*(.*)",
        "severity": r"Severity:\s*(.*)",
        "specialist": r"Specialist:\s*(.*)",
        "advice": r"Detailed Advice:\s*(.*)",
        "when_to_seek_help": r"When to Seek Help:\s*(.*)",
        "cause": r"Cause:\s*(.*)"
    }

    for key, pat in patterns.items():
        m = re.search(pat, txt, re.IGNORECASE)
        if m:
            data[key] = m.group(1).strip()

    return data

# ------------------ Prompt Templates ------------------
start_template = """[INST] <<SYS>>
You are a professional doctor.
The patient points to: "{area}".
TASK: Ask ONE short question only.
RULES: No greetings, only the question.
<</SYS>>
Doctor:"""

investigator_template = """[INST] <<SYS>>
Role: Doctor. Turn: {turn}/8.
Task: Ask ONE short follow-up question.
If enough info output EXACTLY: READY_TO_DIAGNOSE
<</SYS>>
History:
{history}
Patient: {input}
Doctor:"""

# ✅ FINAL REPORT TEMPLATE
report_template = """[INST] <<SYS>>
You are a medical AI. Output the FINAL REPORT strictly in this format:

Disease: ...
Immediate Action: ...
Self-Care: ...
Severity: ...
Specialist: ...
Detailed Advice: ...
When to Seek Help: ...
Cause: ...

Rules:
- Write LONG detailed helpful text for each line
- No medications
- No extra commentary outside these fields
<</SYS>>

History:
{history}

Doctor: Generate Report
"""

# ------------------ History Render ------------------
def render_history(session_id: str) -> str:
    parts = []
    for m in store[session_id]:
        role = m["role"]
        txt = m["text"]
        parts.append(f"{role.capitalize()}: {txt}")
    return "\n".join(parts)

# ------------------ Generation Wrapper ------------------
def run_generation(prompt: str, max_new_tokens=256) -> str:
    out = gen_pipeline(prompt, max_new_tokens=max_new_tokens)
    return out[0]["generated_text"]

# ------------------ Report Generator ------------------
def generate_report(session_id: str):
    history_txt = render_history(session_id)
    prompt = report_template.format(history=history_txt)
    raw = run_generation(prompt, max_new_tokens=512)
    parsed = extract_report_data(raw)
    return {
        "status": "report",
        "content": "Diagnosis Ready",
        "report_data": parsed
    }

# ------------------ Medical Brain Logic ------------------
def medical_brain(user_input: str, session_id: str):
    count = get_question_count(session_id)

    # First Question
    if count == 0:
        increment_question_count(session_id)
        prompt = start_template.format(area=user_input)
        q = strict_clean_response(run_generation(prompt))

        store[session_id].append({"role": "patient", "text": user_input})
        store[session_id].append({"role": "doctor", "text": q})

        return {"status": "question", "content": q}

    # Follow-up
    increment_question_count(session_id)
    current_turn = get_question_count(session_id)

    history_txt = render_history(session_id)
    prompt = investigator_template.format(
        turn=current_turn,
        history=history_txt,
        input=user_input
    )
    q = strict_clean_response(run_generation(prompt))

    store[session_id].append({"role": "patient", "text": user_input})
    store[session_id].append({"role": "doctor", "text": q})

    # If model signals READY_TO_DIAGNOSE or last question, generate report
    if q == "READY_TO_DIAGNOSE" or current_turn >= MAX_QUESTIONS:
        return generate_report(session_id)

    return {"status": "question", "content": q}

# ------------------ FastAPI Server ------------------
app = FastAPI(title="Medical AI Server")

# ✅ هذا الجزء الجديد الوحيد — يسمح للمتصفح بالاتصال بالسيرفر
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class ChatRequest(BaseModel):
    session_id: str
    message: str

@app.post("/chat")
def chat(req: ChatRequest):
    return medical_brain(req.message, req.session_id)

# ------------------ Ngrok Tunnel ------------------
NGROK_AUTH_TOKEN = "38YIqmgsWHFwzD4ykQoFV4bhOuF_4houKhQZVgCSJncunqURL"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

try:
    ngrok.kill()
except:
    pass

public_url = ngrok.connect(PORT).public_url
print("🚀 Public URL:", public_url)
print("📄 Docs:", public_url + "/docs")

# ------------------ Run Server ------------------
def run():
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=PORT)

threading.Thread(target=run, daemon=True).start()
print("✅ Server Running")
